# 🚌 IQueue — Fine-tune XLM-RoBERTa for Multilingual Intent Classification

Fine-tunes `FacebookAI/xlm-roberta-base` on 5 intents across 4 ASEAN languages.

### Quick Start
1. **Upload** `iqueue_train.csv` and `iqueue_val.csv`
2. **Enable GPU** — Runtime → Change runtime type → T4 GPU (Colab) or Accelerator → GPU P100 (Kaggle)
3. **Run all cells** (Runtime → Run all)

### Output
- `xlm-roberta-iqueue.zip` — trained model artifact (~1.1 GB)

In [ ]:
# ============================================================================
# 0. Environment Setup
# ============================================================================

import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle" in sys.modules if hasattr(sys, "modules") else False

print(f"Environment: {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")

In [ ]:
# Install dependencies (idempotent — skips already-installed packages)
!pip install -q transformers datasets scikit-learn pandas numpy accelerate

In [ ]:
# ============================================================================
# 1. Upload CSVs
# ============================================================================
#
# Colab   → file picker opens automatically
# Kaggle  → add CSVs via the "+ Add Data" button, then set KAGGLE_DATA_PATH below
# Local   → place CSVs next to this notebook

KAGGLE_DATA_PATH = "/kaggle/input/datasets/aaronjalapon1/iqueue"  # ← edit for your Kaggle dataset

if IS_COLAB:
    from google.colab import files
    print("📎 Upload iqueue_train.csv and iqueue_val.csv:")
    uploaded = files.upload()
    DATA_DIR = Path(".")
elif IS_KAGGLE:
    DATA_DIR = Path(KAGGLE_DATA_PATH)
    print(f"📁 Using Kaggle dataset at: {DATA_DIR}")
    print(f"   Files found: {sorted(f.name for f in DATA_DIR.iterdir())}")
else:
    DATA_DIR = Path(".")
    print("💻 Local mode — ensure CSVs are in the working directory")

In [ ]:
# ============================================================================
# 2. Imports & Configuration
# ============================================================================

import json
import shutil
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

# ---------------------------------------------------------------------------
# Configuration — edit these if needed
# ---------------------------------------------------------------------------

MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LENGTH = 128
OUTPUT_DIR = Path("./xlm-roberta-iqueue")

EPOCHS = 5
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
SEED = 42

LABELS = [
    "check_booking",
    "request_requeue",
    "get_departure_info",
    "surge_info",
    "fallback",
]
LABEL2ID = {lbl: i for i, lbl in enumerate(LABELS)}
ID2LABEL = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

print(f"PyTorch  : {torch.__version__}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ None (training will be slow!)'}")
print(f"Labels   : {LABELS}")

In [ ]:
# ============================================================================
# 3. Load & Validate Data
# ============================================================================

def load_datasets(data_dir: Path) -> DatasetDict:
    """Load train / val / test CSVs and convert to Hugging Face Datasets."""
    ds_dict = {}

    # --- Train ---
    train_path = data_dir / "iqueue_train.csv"
    if not train_path.exists():
        raise FileNotFoundError(f"Train CSV not found at {train_path}")

    train_df = pd.read_csv(train_path)
    train_df["label_id"] = train_df["label"].map(LABEL2ID)
    ds_dict["train"] = Dataset.from_pandas(
        train_df[["text", "label_id", "language", "source"]]
    )

    # --- Validation ---
    val_path = data_dir / "iqueue_val.csv"
    if val_path.exists():
        val_df = pd.read_csv(val_path)
        val_df["label_id"] = val_df["label"].map(LABEL2ID)
        ds_dict["validation"] = Dataset.from_pandas(
            val_df[["text", "label_id", "language", "source"]]
        )
    else:
        print("⚠ No validation CSV — training without eval")

    return DatasetDict(ds_dict)


ds = load_datasets(DATA_DIR)

print(f"\nTrain examples : {len(ds['train'])}")
print(f"Val examples   : {len(ds.get('validation', []))}")

# Label distribution
train_labels = ds["train"]["label_id"]
label_counts = {LABELS[i]: train_labels.count(i) for i in range(NUM_LABELS)}
print(f"\nLabel distribution:")
for lbl, cnt in label_counts.items():
    bar = "█" * (cnt // 10)
    print(f"  {lbl:25s} : {cnt:4d}  {bar}")

# Check for rare labels
rare_labels = [lbl for lbl, cnt in label_counts.items() if cnt < 5]
if rare_labels:
    raise ValueError(f"Labels with <5 examples: {rare_labels}. Add more data!")

# Show samples
print("\n📋 Sample rows:")
for i in range(min(5, len(ds["train"]))):
    ex = ds["train"][i]
    print(f"  [{ex['language']}] {ex['text'][:90]}…  →  {LABELS[ex['label_id']]}")

In [ ]:
# ============================================================================
# 4. Load Model & Tokenizer
# ============================================================================

print(f"📥 Loading {MODEL_NAME} …")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print(f"   Parameters: {model.num_parameters():,}")
print(f"   (CLASSIFIER weights are NEW — will be learned during training)")

In [ ]:
# ============================================================================
# 5. Tokenize
# ============================================================================

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


print(f"🔪 Tokenizing (max_length={MAX_LENGTH}) …")
tokenized = ds.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label_id", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("   Done.")

In [ ]:
# ============================================================================
# 6. Train
# ============================================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": round(acc, 4)}


has_val = "validation" in ds

# Compute warmup steps (avoids deprecated warmup_ratio)
total_steps = (len(tokenized["train"]) // BATCH_SIZE) * EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch" if has_val else "no",
    save_strategy="epoch",
    load_best_model_at_end=True if has_val else False,
    metric_for_best_model="accuracy" if has_val else None,
    logging_steps=10,
    warmup_steps=warmup_steps,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    save_total_limit=2,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized.get("validation"),
    compute_metrics=compute_metrics,
)

print(f"🚀 Starting training")
print(f"   Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print(f"   Warmup steps: {warmup_steps}  |  Total steps: {total_steps}")
print(f"   FP16: {torch.cuda.is_available()}")
print()
trainer.train()

In [ ]:
# ============================================================================
# 7. Evaluate on Validation Set
# ============================================================================

if has_val:
    metrics = trainer.evaluate()
    print(f"📊 Validation: {metrics}")
else:
    print("⚠ No validation set — skipping eval.")

In [ ]:
# ============================================================================
# 8. Save Model & Label Map
# ============================================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"💾 Model saved → {OUTPUT_DIR}")
print(f"   Files: {sorted(f.name for f in OUTPUT_DIR.iterdir())}")

# Label map (JSON with string keys)
label_map_path = OUTPUT_DIR / "label_map.json"
with open(label_map_path, "w") as f:
    json.dump({str(k): v for k, v in ID2LABEL.items()}, f, indent=2)
metadata = {"labels": LABELS, "model": MODEL_NAME, "max_length": MAX_LENGTH}
with open(OUTPUT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"📋 Label map → {label_map_path}")

In [ ]:
# ============================================================================
# 9. Download Model Artifact
# ============================================================================

ZIP_NAME = "xlm-roberta-iqueue.zip"
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
zip_path = Path.cwd() / ZIP_NAME
print(f"📦 Archive: {ZIP_NAME} ({zip_path.stat().st_size / 1e6:.0f} MB)")

if IS_COLAB:
    from google.colab import files
    files.download(str(zip_path))
elif IS_KAGGLE:
    print("   Download from the Output tab (right sidebar)")
else:
    print(f"   Ready at {zip_path}")

---

### ✅ Done! Next Steps

1. **Download** `xlm-roberta-iqueue.zip`
2. **Unzip** into your project:
   ```bash
   unzip xlm-roberta-iqueue.zip -d ml/chatbot/artifacts/xlm-roberta-iqueue/
   ```
3. **Evaluate** — open `evaluate.ipynb` or run `python ml/chatbot/evaluate.py`